In [1]:
!pip install ucimlrepo

In [7]:
!pip install opacus

In [6]:
from datasets import load_dataset

ds = load_dataset("ylecun/mnist")

README.md:   0%|          | 0.00/6.97k [00:00<?, ?B/s]

mnist/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.6MB            

mnist/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mnist/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.60MB            

mnist/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [9]:
ds

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [10]:
import copy, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
from datasets import load_dataset
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)
DEVICE = "cpu"
DELTA, MAX_GRAD_NORM, BATCH, EPOCHS = 1e-5, 1.0, 256, 3

# ---- Load ylecun/mnist from Hugging Face ----
ds = load_dataset("ylecun/mnist")          # splits: 'train' (60000), 'test' (10000)
print(ds)

MEAN, STD = 0.1307, 0.3081

def to_tensors(split, n=None):
    split = split if n is None else split.select(range(n))
    imgs = np.stack([np.array(im, dtype=np.uint8) for im in split["image"]])   # (N,28,28)
    X = torch.from_numpy(imgs).float().div(255.0)                              # scale to [0,1]
    X = ((X - MEAN) / STD).unsqueeze(1)                                        # (N,1,28,28)
    y = torch.tensor(split["label"], dtype=torch.long)
    return X, y

Xtr, ytr = to_tensors(ds["train"], n=20000)   # 20,000-image subset
Xte, yte = to_tensors(ds["test"])             # full 10,000 test images

train_set = TensorDataset(Xtr, ytr)
test_set  = TensorDataset(Xte, yte)

train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=1000)
print("train:", len(train_set), "test:", len(test_set), "| image shape:", tuple(Xtr.shape[1:]))

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})
train: 20000 test: 10000 | image shape: (1, 28, 28)


In [11]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 16, 3, padding=1)
        self.g1 = nn.GroupNorm(4, 16)          # GroupNorm (Opacus-compatible)
        self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.g2 = nn.GroupNorm(8, 32)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.g1(self.c1(x))), 2)   # 14x14
        x = F.max_pool2d(F.relu(self.g2(self.c2(x))), 2)   # 7x7
        x = x.flatten(1)
        return self.fc2(F.relu(self.fc1(x)))

def new_model(seed=42):
    torch.manual_seed(seed)
    m = CNN()
    assert ModuleValidator.is_valid(m)         # confirms Opacus compatibility
    return m.to(DEVICE)

def train_epoch(model, loader, opt):
    model.train()
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        F.cross_entropy(model(x), y).backward()
        opt.step()

@torch.no_grad()
def accuracy(model, loader):
    model.eval(); correct = 0
    for x, y in loader:
        correct += (model(x).argmax(1) == y).sum().item()
    return 100.0 * correct / len(loader.dataset)

@torch.no_grad()
def mean_confidence(model, dataset, n=5000):
    """Mean max-softmax confidence over the first n samples."""
    model.eval()
    loader = DataLoader(Subset(dataset, range(min(n, len(dataset)))), batch_size=1000)
    confs = [F.softmax(model(x), 1).max(1).values for x, _ in loader]
    return torch.cat(confs).mean().item()

In [12]:
torch.manual_seed(42)
baseline = new_model()
opt = torch.optim.SGD(baseline.parameters(), lr=0.1, momentum=0.9)
for ep in range(EPOCHS):
    train_epoch(baseline, train_loader, opt)
    print(f"Baseline epoch {ep+1}: test acc = {accuracy(baseline, test_loader):.2f}%")
base_acc = accuracy(baseline, test_loader)
print(f"\nBaseline (no DP) : test accuracy {base_acc:.1f} %")

Baseline epoch 1: test acc = 95.34%
Baseline epoch 2: test acc = 97.81%
Baseline epoch 3: test acc = 97.35%

Baseline (no DP) : test accuracy 97.3 %


In [13]:
def train_dp_target_eps(target_eps, lr=0.5):
    torch.manual_seed(42)
    model = new_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
    pe = PrivacyEngine()
    model, opt, loader = pe.make_private_with_epsilon(
        module=model, optimizer=opt, data_loader=loader,
        epochs=EPOCHS, target_epsilon=target_eps,
        target_delta=DELTA, max_grad_norm=MAX_GRAD_NORM)
    for ep in range(EPOCHS):
        train_epoch(model, loader, opt)
    eps_spent = pe.get_epsilon(DELTA)
    return model, accuracy(model, test_loader), eps_spent

results, dp_models = {}, {}
for eps in [10, 5, 1]:
    m, acc, spent = train_dp_target_eps(eps)
    results[eps] = (acc, spent); dp_models[eps] = m
    print(f"eps={eps}: acc={acc:.1f}%  eps_spent={spent:.2f}")

print("\nTARGET_EPS   DELTA    CLIP(C)   EPOCHS   TEST_ACC   EPS_SPENT")
print("-" * 61)
for eps in [10, 5, 1]:
    acc, spent = results[eps]
    print(f"{eps:>7.1f}   {DELTA:>8.0e}   {MAX_GRAD_NORM:>6.1f}   {EPOCHS:>6d}   {acc:>7.1f} %   {spent:>8.2f}")
print("-" * 61)
print(f"Accuracy cost of strongest privacy (eps = 1) : {results[1][0]-base_acc:+.1f} %")

eps=10: acc=95.2%  eps_spent=9.99
eps=5: acc=94.2%  eps_spent=5.00
eps=1: acc=88.4%  eps_spent=1.00

TARGET_EPS   DELTA    CLIP(C)   EPOCHS   TEST_ACC   EPS_SPENT
-------------------------------------------------------------
   10.0      1e-05      1.0        3      95.2 %       9.99
    5.0      1e-05      1.0        3      94.2 %       5.00
    1.0      1e-05      1.0        3      88.4 %       1.00
-------------------------------------------------------------
Accuracy cost of strongest privacy (eps = 1) : -9.0 %


In [14]:
torch.manual_seed(42)
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.5, momentum=0.9)
loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
pe = PrivacyEngine()
model, opt, loader = pe.make_private(
    module=model, optimizer=opt, data_loader=loader,
    noise_multiplier=1.1, max_grad_norm=MAX_GRAD_NORM)

eps_per_epoch = []
print("FIXED noise_multiplier = 1.1\nEPOCH   EPSILON_SPENT\n" + "-" * 21)
for ep in range(EPOCHS):
    train_epoch(model, loader, opt)
    e = pe.get_epsilon(DELTA)
    eps_per_epoch.append(e)
    print(f"  {ep+1}         {e:.2f}")
print("-" * 21)
print(f"Test acc (sigma=1.1): {accuracy(model, test_loader):.1f} %")

FIXED noise_multiplier = 1.1
EPOCH   EPSILON_SPENT
---------------------
  1         0.66
  2         0.85
  3         1.01
---------------------
Test acc (sigma=1.1): 88.4 %


In [15]:
# Train samples = the 20k subset; test samples = the MNIST test set.
# Both use the same sample size so the comparison is fair.
def gap_row(name, m):
    tr = mean_confidence(m, train_set)
    te = mean_confidence(m, test_set)
    return name, tr, te, tr - te

# Opacus wraps the model in GradSampleModule; it still works for inference.
rows = [gap_row("Baseline", baseline), gap_row("DP (eps=1)", dp_models[1])]
print("MEMBERSHIP CHECK (mean max-softmax confidence)")
print("MODEL        TRAIN    TEST     GAP\n" + "-" * 35)
for n, tr, te, g in rows:
    print(f"{n:<12} {tr:.3f}    {te:.3f}    {g:.3f}")
print("-" * 35)

MEMBERSHIP CHECK (mean max-softmax confidence)
MODEL        TRAIN    TEST     GAP
-----------------------------------
Baseline     0.980    0.972    0.007
DP (eps=1)   0.980    0.977    0.003
-----------------------------------


In [16]:
eps_list = [1, 5, 10]
accs = [results[e][0] for e in eps_list]

# Plot 1: privacy-utility curve
plt.figure(figsize=(6, 4))
plt.plot(eps_list, accs, "o-", label="DP-SGD")
plt.axhline(base_acc, color="r", ls="--", label=f"Baseline ({base_acc:.1f}%)")
for e, a in zip(eps_list, accs):
    plt.annotate(f"{a:.1f}%", (e, a), textcoords="offset points", xytext=(0, 8), ha="center")
plt.xlabel("Target epsilon (ε)"); plt.ylabel("Test accuracy (%)")
plt.title("Privacy-Utility Trade-off"); plt.grid(alpha=.3); plt.legend()
plt.tight_layout(); plt.savefig("accuracy_vs_epsilon.png", dpi=150); plt.close()

# Plot 2: epsilon vs epoch
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), eps_per_epoch, "s-", color="g")
plt.xticks(range(1, EPOCHS + 1))
plt.xlabel("Epoch"); plt.ylabel("ε spent (δ=1e-5)")
plt.title("Privacy budget vs epoch (noise_multiplier = 1.1)")
plt.grid(alpha=.3); plt.tight_layout()
plt.savefig("epsilon_vs_epoch.png", dpi=150); plt.close()
print("Plots saved: accuracy_vs_epsilon.png, epsilon_vs_epoch.png")

Plots saved: accuracy_vs_epsilon.png, epsilon_vs_epoch.png


In [17]:
print("\n================ FINAL RESULTS ================")
print(f"{'Model':<16}{'ε spent':>9}{'Test acc':>11}{'Train conf':>12}{'Test conf':>11}{'Gap':>8}")
print(f"{'Baseline':<16}{'∞':>9}{base_acc:>10.1f}%{rows[0][1]:>12.3f}{rows[0][2]:>11.3f}{rows[0][3]:>8.3f}")
for e in [10, 5, 1]:
    a, s = results[e]
    extra = f"{rows[1][1]:>12.3f}{rows[1][2]:>11.3f}{rows[1][3]:>8.3f}" if e == 1 else ""
    print(f"{'DP ε='+str(e):<16}{s:>9.2f}{a:>10.1f}%{extra}")
print(f"\nσ=1.1 run, ε per epoch: {[round(x,2) for x in eps_per_epoch]}")
print("Recommended ε : 1  (strongest privacy for a small accuracy loss)")


================ FINAL RESULTS ================
Model             ε spent   Test acc  Train conf  Test conf     Gap
Baseline                ∞      97.3%       0.980      0.972   0.007
DP ε=10              9.99      95.2%
DP ε=5               5.00      94.2%
DP ε=1               1.00      88.4%       0.980      0.977   0.003

σ=1.1 run, ε per epoch: [np.float64(0.66), np.float64(0.85), np.float64(1.01)]
Recommended ε : 1  (strongest privacy for a small accuracy loss)


In [18]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [19]:
!git remote add origin https://github.com/FireFistisDead/23AIML027_External_prac

In [20]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	accuracy_vs_epsilon.png
	epsilon_vs_epoch.png
	sample_data/
	t10k-images.idx3-ubyte
	t10k-labels.idx1-ubyte
	train-images.idx3-ubyte
	train-labels.idx1-ubyte

nothing added to commit but untracked files present (use "git add" to track)


In [21]:
!git add accuracy_vs_epsilon.png epsilon_vs_epoch.png

In [ ]:
!git add